In [ ]:
import os, sys, glob, time, shlex, shutil, subprocess, textwrap, json, pathlib

API_KEY   = ""
API_BASE  = "https://api.openai.com/v1"
TEXT_MODEL  = "gpt-4o"
IMAGE_MODEL = "gpt-image-1"

RUN_FIGURE_TECHROUTE = True
RUN_FIGURE_MODELARCH = True
RUN_FIGURE_EXPDATA   = True
RUN_PAPER2PPT        = True
RUN_PPTPOLISH        = True

RUN_PDF_DEMO         = False
ARXIV_PDF_URL        = "https://arxiv.org/pdf/1706.03762"
MINERU_API_KEY       = ""
MINERU_API_BASE_URL  = "https://mineru.net/api/v4"

REPO_DIR             = "/content/Paper2Any"
OUT_ROOT             = "/content/p2a_outputs"
SKIP_INSTALL_IF_PRESENT = True

if not API_KEY:
    try:
        from getpass import getpass
        API_KEY = getpass("🔑 Enter your OpenAI-compatible API key (sk-...): ").strip()
    except Exception:
        pass
if not API_KEY:
    print("⚠️  No API key provided. Install steps will still run, but generation will fail.")

os.environ.update({
    "DF_API_KEY": API_KEY, "DF_API_URL": API_BASE, "DF_MODEL": TEXT_MODEL,
    "OPENAI_API_KEY": API_KEY, "OPENAI_BASE_URL": API_BASE,
})
if MINERU_API_KEY:
    os.environ.update({"MINERU_API_KEY": MINERU_API_KEY,
                       "MINERU_API_BASE_URL": MINERU_API_BASE_URL})

os.makedirs(OUT_ROOT, exist_ok=True)

In [ ]:
def sh(cmd, cwd=None, env=None, stream=True, check=False):
    if stream:
        print(f"\n$ {cmd}", flush=True)
    proc = subprocess.Popen(cmd, shell=True, cwd=cwd,
                            env={**os.environ, **(env or {})},
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    buf = []
    for line in proc.stdout:
        buf.append(line)
        if stream:
            print(line, end="", flush=True)
    proc.wait()
    out = "".join(buf)
    if check and proc.returncode != 0:
        raise RuntimeError(f"Command failed ({proc.returncode}): {cmd}\n{out[-2000:]}")
    return proc.returncode, out

def banner(title):
    print("\n" + "═" * 92 + f"\n  {title}\n" + "═" * 92, flush=True)

def have(tool):
    return shutil.which(tool) is not None

def cli_help(script):
    _, out = sh(f"python {script} --help", cwd=REPO_DIR, stream=False)
    return out

def find_newest(roots, patterns, since_ts=0.0):
    hits = []
    roots = [r for r in roots if r and os.path.isdir(r)]
    for r in roots:
        for pat in patterns:
            for f in glob.glob(os.path.join(r, "**", pat), recursive=True):
                try:
                    if os.path.getmtime(f) >= since_ts - 1:
                        hits.append(f)
                except OSError:
                    pass
    return sorted(set(hits), key=lambda f: os.path.getmtime(f))

try:
    from IPython.display import Image as _IPyImage, display as _ipy_display
    def show_img(path, w=720):
        try:
            _ipy_display(_IPyImage(filename=path, width=w))
        except Exception as e:
            print("   (could not render", path, "->", e, ")")
except Exception:
    def show_img(path, w=720):
        print("   preview:", path)

def pptx_to_pngs(pptx_path, work_dir, max_pages=2, dpi=110):
    os.makedirs(work_dir, exist_ok=True)
    if not have("soffice") and not have("libreoffice"):
        return []
    soffice = "soffice" if have("soffice") else "libreoffice"
    sh(f'{soffice} --headless -env:UserInstallation=file:///tmp/lo_profile '
       f'--convert-to pdf --outdir {shlex.quote(work_dir)} {shlex.quote(pptx_path)}', stream=False)
    pdf = os.path.join(work_dir, os.path.splitext(os.path.basename(pptx_path))[0] + ".pdf")
    if not os.path.exists(pdf):
        return []
    stem = os.path.join(work_dir, "slide")
    sh(f"pdftoppm -png -r {dpi} -l {max_pages} {shlex.quote(pdf)} {shlex.quote(stem)}", stream=False)
    return sorted(glob.glob(stem + "*.png"))

def svg_to_png(svg_path, png_path):
    if have("inkscape"):
        rc, _ = sh(f"inkscape {shlex.quote(svg_path)} --export-type=png "
                   f"--export-filename={shlex.quote(png_path)} -w 1200", stream=False)
        if rc == 0 and os.path.exists(png_path):
            return png_path
    try:
        import cairosvg
        cairosvg.svg2png(url=svg_path, write_to=png_path, output_width=1200)
        return png_path
    except Exception:
        return None

In [ ]:
banner("STEP 1/6 · System dependencies (LibreOffice, Inkscape, Poppler, ffmpeg, tectonic)")

need_apt = not all(have(t) for t in ["soffice", "inkscape", "pdftoppm", "ffmpeg"])
if need_apt or not SKIP_INSTALL_IF_PRESENT:
    sh("apt-get -qq update")
    sh("apt-get -qq install -y --no-install-recommends "
       "libreoffice inkscape poppler-utils wkhtmltopdf ffmpeg "
       "fonts-dejavu fonts-liberation fonts-noto-cjk")
else:
    print("System tools already present — skipping apt.")

if not have("tectonic"):
    print("Installing tectonic (LaTeX engine) ...")
    rc, _ = sh("cd /usr/local/bin && "
               "curl --proto '=https' --tlsv1.2 -fsSL https://drop-sh.fullyjustified.net | sh")
    if not have("tectonic"):
        sh("apt-get -qq install -y tectonic || true")
print("tectonic:", shutil.which("tectonic") or "NOT FOUND (LaTeX-rendered figures may be limited)")

banner("STEP 2/6 · Clone Paper2Any and install Python dependencies")

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    sh(f"git clone --depth 1 https://github.com/OpenDCAI/Paper2Any.git {REPO_DIR}", check=True)
else:
    print("Repo already cloned.")

def deps_installed():
    rc, _ = sh('python -c "import dataflow_agent"', cwd=REPO_DIR, stream=False)
    return rc == 0

if deps_installed() and SKIP_INSTALL_IF_PRESENT:
    print("dataflow_agent already importable — skipping pip install.")
else:
    sh("pip install -q -r requirements-base.txt", cwd=REPO_DIR)
    sh("pip install -q -e .", cwd=REPO_DIR)
    rc, _ = sh("pip install -q -r requirements-paper.txt", cwd=REPO_DIR)
    if rc != 0:
        print("requirements-paper.txt failed -> trying requirements-paper-backup.txt")
        sh("pip install -q -r requirements-paper-backup.txt", cwd=REPO_DIR)
    sh("pip install -q doclayout_yolo --no-deps", cwd=REPO_DIR)
    sh("pip install -q pdf2image PyMuPDF python-pptx pillow cairosvg requests", cwd=REPO_DIR)

banner("STEP 3/6 · Preflight checks")
for t in ["python", "git", "tectonic", "inkscape", "soffice", "pdftoppm", "ffmpeg", "wkhtmltopdf"]:
    print(f"  {t:12s}: {shutil.which(t) or '❌ NOT FOUND'}")
rc, out = sh('python -c "import dataflow_agent, pptx, fitz; print(\'core imports OK\')"',
             cwd=REPO_DIR, stream=False)
print(" ", out.strip() or "❌ core import check failed:")
if rc != 0:
    print(out[-1500:])
print(f"\n  Text model : {TEXT_MODEL}\n  API base   : {API_BASE}\n  API key set : {'yes' if API_KEY else 'NO'}")

In [ ]:
def oneline(s):
    return " ".join(s.split())

TECH_ROUTE_TEXT = oneline("""
A self-improving "data flywheel" pipeline for training instruction-following LLMs. Stage 1:
collect raw web and domain documents. Stage 2: clean, deduplicate, and quality-filter the
corpus. Stage 3: generate synthetic instruction-response pairs and augment with self-instruct.
Stage 4: supervised fine-tuning (SFT) on the curated set. Stage 5: preference optimization with
RLHF / DPO using a learned reward model. Stage 6: automatic evaluation on held-out benchmarks
and red-teaming. Stage 7: deploy, log real user interactions, and feed failures back into Stage
1 to close the loop. Show the seven stages left-to-right with a feedback arrow from evaluation
and deployment back to data collection.
""")

MODEL_ARCH_TEXT = oneline("""
A multimodal transformer for document understanding. Inputs are a page image and an OCR text
sequence. The image path uses a ViT patch encoder; the text path uses a token embedding plus
positional encoding. A cross-attention fusion block lets text queries attend to visual patches.
The fused representation passes through N stacked transformer encoder layers (multi-head
self-attention + feed-forward + residual + layer norm). A task head produces three outputs:
layout boxes, entity labels, and a generated summary via an autoregressive decoder. Annotate
the encoders, the fusion block, the encoder stack, and the three task heads.
""")

EXP_DATA_TEXT = oneline("""
Plot experimental results comparing three training methods across model scale. X-axis is model
size in billions of parameters (7, 13, 70). Y-axis is accuracy (%). Method "Baseline (SFT)":
58.1, 64.7, 71.2. Method "+ Synthetic Augmentation": 61.5, 68.9, 75.8. Method "+ RLHF": 64.3,
72.4, 80.6. Use a grouped line chart with markers, a legend, axis labels, and a title
"Accuracy vs. Model Scale by Training Method".
""")

PPT_TOPIC_TEXT = oneline("""
Title: Paper2Any — A Multi-Agent Workflow for Editable Academic Multimodal Content. Abstract:
We present a system that converts research papers, free text, or a single topic into editable
artifacts: scientific figures, technical-route diagrams, experimental plots, and slide decks.
The pipeline parses the source, lets a language model plan a structured specification, fills the
specification, and renders it with LaTeX, SVG, and PPTX tooling so every output remains editable.
We discuss the agent architecture, the figure-rendering toolchain, evaluation of layout fidelity,
limitations such as PDF-parsing cost, and directions for future work including video and posters.
""")

def run_workflow(name, script, input_value, out_subdir, *,
                 graph_type=None, input_is_text=False, extra_flags=None, use_image_model=False):
    banner(f"RUN · {name}")
    script_path = os.path.join("script", script)
    if not os.path.exists(os.path.join(REPO_DIR, script_path)):
        print(f"⚠️  {script_path} not found in this version of the repo — skipping.")
        return None, 0.0
    help_text = cli_help(script_path)
    out_dir = os.path.join(OUT_ROOT, out_subdir)
    os.makedirs(out_dir, exist_ok=True)

    parts = ["python", script_path, "--input", shlex.quote(input_value)]

    def add(flag, value=None):
        if flag in help_text:
            parts.append(flag)
            if value is not None:
                parts.append(shlex.quote(str(value)))
            return True
        return False

    add("--api-key", API_KEY)
    add("--api-url", API_BASE)
    add("--model", TEXT_MODEL)
    add("--output-dir", out_dir)
    if graph_type:
        add("--graph-type", graph_type)
    if input_is_text:
        add("--input-type", "TEXT")
    if use_image_model and IMAGE_MODEL:
        for f in ("--image-model", "--img-model", "--image_model"):
            if add(f, IMAGE_MODEL):
                break
    for f, v in (extra_flags or []):
        add(f, v)

    cmd = " ".join(parts)
    start = time.time()
    env = {"DF_API_KEY": API_KEY, "DF_API_URL": API_BASE, "DF_MODEL": TEXT_MODEL}
    rc, _ = sh(cmd, cwd=REPO_DIR, env=env, stream=True)
    dur = time.time() - start
    print(f"\n[{name}] exit={rc}  elapsed={dur:.0f}s")
    return out_dir, start

In [1]:
banner("STEP 4/6 · Generate artifacts")
run_started_at = time.time()
produced_dirs = []

if RUN_FIGURE_TECHROUTE:
    d, _ = run_workflow("Paper2Figure · Technical Route", "run_paper2figure_cli.py",
                        TECH_ROUTE_TEXT, "figure_tech_route",
                        graph_type="tech_route", input_is_text=True, use_image_model=True)
    produced_dirs.append(d)

if RUN_FIGURE_MODELARCH:
    d, _ = run_workflow("Paper2Figure · Model Architecture", "run_paper2figure_cli.py",
                        MODEL_ARCH_TEXT, "figure_model_arch",
                        graph_type="model_arch", input_is_text=True, use_image_model=True)
    produced_dirs.append(d)

if RUN_FIGURE_EXPDATA:
    d, _ = run_workflow("Paper2Figure · Experimental Plot", "run_paper2figure_cli.py",
                        EXP_DATA_TEXT, "figure_exp_data",
                        graph_type="exp_data", input_is_text=True)
    produced_dirs.append(d)

if RUN_PAPER2PPT:
    d, _ = run_workflow("Paper2PPT · Text → Slide Deck", "run_paper2ppt_cli.py",
                        PPT_TOPIC_TEXT, "paper2ppt", input_is_text=True,
                        extra_flags=[("--page-count", "10"),
                                     ("--style", "Academic style; English; modern, clean, high-contrast"),
                                     ("--language", "en")])
    produced_dirs.append(d)

if RUN_PPTPOLISH:
    deck = find_newest([os.path.join(OUT_ROOT, "paper2ppt"), os.path.join(REPO_DIR, "outputs")],
                       ["*.pptx"], since_ts=run_started_at)
    if deck:
        src = deck[-1]
        print(f"Polishing newest deck: {src}")
        run_workflow("PPTPolish · Beautify Deck", "run_ppt2polish_cli.py",
                     src, "ppt_polish",
                     extra_flags=[("--style", "Academic style, clean and elegant, consistent spacing")])
        produced_dirs.append(os.path.join(OUT_ROOT, "ppt_polish"))
    else:
        print("⚠️  No .pptx produced by Paper2PPT yet — skipping PPTPolish.")

if RUN_PDF_DEMO:
    banner("RUN · Paper2PPT from a real arXiv PDF (advanced; needs MinerU)")
    pdf_path = os.path.join(OUT_ROOT, "sample_paper.pdf")
    if not os.path.exists(pdf_path):
        sh(f"wget -q -O {shlex.quote(pdf_path)} {shlex.quote(ARXIV_PDF_URL)}")
    if not MINERU_API_KEY:
        print("ℹ️  No MINERU_API_KEY set — PDF parsing will try local MinerU and may be slow "
              "or fail on a CPU runtime. Set MINERU_API_KEY above for a smooth run.")
    run_workflow("Paper2PPT · PDF", "run_paper2ppt_cli.py",
                 pdf_path, "paper2ppt_pdf",
                 extra_flags=[("--page-count", "12"), ("--language", "en")])
    produced_dirs.append(os.path.join(OUT_ROOT, "paper2ppt_pdf"))

banner("STEP 5/6 · Preview generated artifacts")

search_roots = [OUT_ROOT, os.path.join(REPO_DIR, "outputs")]
pptx_files = find_newest(search_roots, ["*.pptx"], since_ts=run_started_at)
svg_files  = find_newest(search_roots, ["*.svg"],  since_ts=run_started_at)
png_files  = find_newest(search_roots, ["*.png", "*.jpg", "*.jpeg"], since_ts=run_started_at)
other      = find_newest(search_roots, ["*.drawio", "*.pdf", "*.mp4", "*.md"], since_ts=run_started_at)

preview_dir = os.path.join(OUT_ROOT, "_previews")
os.makedirs(preview_dir, exist_ok=True)

if pptx_files:
    print(f"\n🖼  {len(pptx_files)} PPTX file(s): rendering first slides ...")
    for p in pptx_files:
        print(f"\n• {p}")
        for img in pptx_to_pngs(p, os.path.join(preview_dir, pathlib.Path(p).stem))[:2]:
            show_img(img)

if svg_files:
    print(f"\n🖼  {len(svg_files)} SVG file(s):")
    for s in svg_files:
        print(f"\n• {s}")
        png = svg_to_png(s, os.path.join(preview_dir, pathlib.Path(s).stem + ".png"))
        if png:
            show_img(png)

if png_files:
    print(f"\n🖼  {len(png_files)} raster image(s) (e.g. experimental plots):")
    for img in png_files:
        print(f"\n• {img}")
        show_img(img)

if not (pptx_files or svg_files or png_files):
    print("No artifacts were found. Check the logs above — the most common cause is a missing or "
          "invalid API key, an unsupported model name, or a model without the needed capability.")

banner("STEP 6/6 · Package everything for download")
all_artifacts = sorted(set(pptx_files + svg_files + png_files + other))
print("Artifacts produced:")
for f in all_artifacts:
    try:
        print(f"  {os.path.getsize(f)/1024:8.1f} KB   {f}")
    except OSError:
        pass

zip_base = "/content/paper2any_outputs"
if os.path.isdir(OUT_ROOT) and os.listdir(OUT_ROOT):
    shutil.make_archive(zip_base, "zip", OUT_ROOT)
    print(f"\n📦 Zipped to {zip_base}.zip")
    try:
        from google.colab import files as _colab_files
        _colab_files.download(zip_base + ".zip")
    except Exception:
        print("   (Not in Colab — grab the zip from the file browser.)")

print("\n✅ Done. Open the .pptx in PowerPoint/Google Slides and the .svg in Inkscape/draw.io — "
      "every element stays editable.")

🔑 Enter your OpenAI-compatible API key (sk-...): ··········

════════════════════════════════════════════════════════════════════════════════════════════
  STEP 1/6 · System dependencies (LibreOffice, Inkscape, Poppler, ffmpeg, tectonic)
════════════════════════════════════════════════════════════════════════════════════════════

$ apt-get -qq update
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)

$ apt-get -qq install -y --no-install-recommends libreoffice inkscape poppler-utils wkhtmltopdf ffmpeg fonts-dejavu fonts-liberation fonts-noto-cjk
Preconfiguring packages ...
Selecting previously unselected package libfftw3-double3:amd64.
(Reading database ... 
(Reading database ... 5%
(Reading database ... 10%
(Reading database ... 15%
(Reading database ... 20%
(Reading database ... 25%
(Reading database ... 30%
(Reading database ... 35%
(Reading data

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Done. Open the .pptx in PowerPoint/Google Slides and the .svg in Inkscape/draw.io — every element stays editable.
